# Transcriptome-wide trans-fit parameter comparison

Compares fitted trans parameters (observed log2FC, EC50, Hill coefficient, ...) between two bayesDREAM runs, gene by gene, for every cis gene both runs have completed a `fit_trans` for.

Unlike `dose_response_comparison.ipynb` (which reloads full models to draw per-gene dose-response curves, and only scales to ~100 genes), this notebook reads each run's `trans_feature_summary_gene.csv` directly -- so it scales to Morris/Replogle's transcriptome-wide trans gene sets. See `comparative/trans_param_compare.py` for the underlying library.

Starts with the simplest comparison (observed log2FC, Morris on x vs Replogle on y), then a grid of other parameters colored by dependency category (dependent in Morris only / Replogle only / both) -- genes dependent in neither are dropped from the grid entirely. Replogle is much less powered than Morris, so points that are `is_dependent` in Morris are outlined on the starting-point plot -- watch where those land in Replogle's noisier axis.

In [ ]:
# run "pip install ipython-autotime" in your conda env
%load_ext autotime

import os
import sys

# Derive the repo root from bayesDREAM's installed location (pip -e .), not
# from os.getcwd() -- the notebook's cwd at kernel start isn't guaranteed to
# be its own directory (depends on how Jupyter/the IDE was launched), so a
# '../..'-from-cwd guess silently fails to find comparative/ in that case.
# importlib.import_module (not a plain 'import bayesDREAM as ...') deliberately
# sidesteps a name collision: the bayesDREAM PACKAGE and the bayesDREAM CLASS it
# exports both share the literal name 'bayesDREAM'. If this cell's import ever
# gets merged with a 'from bayesDREAM import bayesDREAM' line elsewhere in the
# notebook, a plain import here can end up aliasing the class (no __file__)
# instead of the package -- this form can't be shadowed that way.
import importlib
_bayesdream_pkg = importlib.import_module('bayesDREAM')
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(_bayesdream_pkg.__file__)))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import matplotlib.pyplot as plt
import pandas as pd

from comparative.datasets import DOMINGO, MORRIS, REPLOGLE, ALL_DATASETS
from comparative.trans_param_compare import (
    load_trans_summary,
    merge_pair,
    plot_obs_log2fc,
    plot_param_grid,
    plot_param_grid_by_dependency,
    compare_cis_gene,
    compare_all_shared_cis_genes,
    compare_cis_gene_grid,
    plot_pairwise_grid,
    compare_all_cis_genes_grid,
    load_all,
    DEFAULT_GRID_PARAMS,
)

## Config

In [ ]:
# Which two datasets to compare, and in which x/y order (matters for the
# scatter plots and for which dataset's is_dependent flag gets highlighted --
# see plot_obs_log2fc/plot_param_grid docstrings, dataset A drives both).
SPEC_X = MORRIS      # x-axis
SPEC_Y = REPLOGLE    # y-axis

PLOT_DIR = './trans_param_comparison_plots'
os.makedirs(PLOT_DIR, exist_ok=True)

# Extra parameters for the grid plot, beyond DEFAULT_GRID_PARAMS -- see
# comparative/trans_param_compare.py's PARAM_ALIASES for what's available.
GRID_PARAMS = DEFAULT_GRID_PARAMS
print('Datasets available:', list(ALL_DATASETS))
print(f'Comparing {SPEC_X.name} (x) vs {SPEC_Y.name} (y)')
print(f'{SPEC_X.name} cis genes with completed fit_trans: {SPEC_X.cis_genes}')
print(f'{SPEC_Y.name} cis genes with completed fit_trans: {SPEC_Y.cis_genes}')

## Backfill (run once per gene, or whenever a fit is re-run)

Reconstructs each dataset's fitted model (from its real production config for Domingo/Morris, from the papermill notebook's own data-loading logic for Replogle -- see `comparative/reconstruct_export.py` / `comparative/reconstruct_export_replogle.py`) and, for every cis gene:

1. backfills `trans_feature_summary_gene.csv` in place with `y_log2fc_at_xm1` (and any other `HILL_LOG2FC_TARGETS`), so it shows up in the comparisons below, and
2. exports it for `dose_response_panels.py`'s full per-gene curve panels via `save_model_for_plotting()`.

This is a full model reload per (dataset, gene) -- slow (Morris/Replogle are transcriptome-wide) -- so it's gated behind `RUN_BACKFILL` below rather than running on every notebook open. Turn it on the first time, and again any time a fit gets re-run.

**Before running this for Replogle**, confirm the assumption documented at the top of `comparative/reconstruct_export_replogle.py`: that all 7 of its `10_bayesDREAM_fit_trans_<GENE>.ipynb` notebooks are the same papermill template (same `INDIR`/`WD`/`NTC_FIT`/`CIS_FIT`/`OUTDIR`, `MIN_LOG2_MU_NTC_TRANS=-4.0`, `function_type="single_hill"`), differing only in the `gene`/`NITERS_TRANS` parameter cell -- that file hardcodes those constants once, for all 7 genes, on that assumption.

In [ ]:
RUN_BACKFILL = False  # set True to (re-)run -- see the note above first

if RUN_BACKFILL:
    from comparative.reconstruct_export import reconstruct_and_export_all
    from comparative.reconstruct_export_replogle import reconstruct_and_export_all as reconstruct_and_export_all_replogle

    dm_results = reconstruct_and_export_all()                 # Domingo + Morris
    replogle_results = reconstruct_and_export_all_replogle()  # Replogle

    print('Domingo/Morris:', {k: list(v) for k, v in dm_results.items()})
    print('Replogle:', list(replogle_results))

## Single cis gene, quick look

Start here with one cis gene before looping over all shared ones below.

In [ ]:
CIS_GENE = 'GFI1B'

df_x = load_trans_summary(SPEC_X, CIS_GENE)
df_y = load_trans_summary(SPEC_Y, CIS_GENE)
merged = merge_pair(df_x, df_y, SPEC_X.name, SPEC_Y.name)
print(f'{SPEC_X.name}: {len(df_x)} trans genes | {SPEC_Y.name}: {len(df_y)} trans genes | shared: {len(merged)}')
merged.head()

### The starting-point plot: observed log2FC

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.6))
plot_obs_log2fc(merged, SPEC_X.name, SPEC_Y.name, CIS_GENE, ax=ax)
fig.savefig(os.path.join(PLOT_DIR, f'{CIS_GENE}_{SPEC_X.name}_vs_{SPEC_Y.name}_obs_log2fc.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### Other parameters, colored by dependency category (genes dependent in neither are dropped)

In [ ]:
fig = plot_param_grid(merged, SPEC_X.name, SPEC_Y.name, CIS_GENE, params=GRID_PARAMS)
fig.savefig(os.path.join(PLOT_DIR, f'{CIS_GENE}_{SPEC_X.name}_vs_{SPEC_Y.name}_param_grid.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### Same parameters, split by dependency category

4 rows -- the 2x2 partition of `is_dependent_{SPEC_X.name}` x `is_dependent_{SPEC_Y.name}`: dependent in `SPEC_X` only, dependent in `SPEC_Y` only, dependent in both, dependent in neither. Useful for seeing whether a parameter's cross-dataset agreement (or disagreement) concentrates in one of these groups -- e.g. "genes both call dependent correlate well, but genes only one dataset calls dependent are basically noise in the other."

In [ ]:
fig = plot_param_grid_by_dependency(merged, SPEC_X.name, SPEC_Y.name, CIS_GENE, params=GRID_PARAMS)
fig.savefig(os.path.join(PLOT_DIR, f'{CIS_GENE}_{SPEC_X.name}_vs_{SPEC_Y.name}_param_grid_by_dependency.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## All shared cis genes

Loops the above over every cis gene present (with a completed fit_trans) in both datasets, saving both plots per gene to `PLOT_DIR`.

In [ ]:
results = compare_all_shared_cis_genes(
    SPEC_X, SPEC_Y,
    out_dir=PLOT_DIR,
    params=GRID_PARAMS,
)
print(f"Done. Wrote plots for: {list(results)}")

## Combine across cis genes

`results` holds one merged DataFrame per cis gene (columns suffixed `_{SPEC_X.name}` / `_{SPEC_Y.name}`) -- concatenate for an across-cis-gene view, e.g. overall correlation or a per-cis-gene breakdown.

In [ ]:
combined = pd.concat(
    [df.assign(cis_gene=gene) for gene, df in results.items()],
    ignore_index=True,
)

fig, ax = plt.subplots(figsize=(5, 5))
for gene, sub in combined.groupby('cis_gene'):
    ax.scatter(sub[f'observed_log2fc_{SPEC_X.name}'], sub[f'observed_log2fc_{SPEC_Y.name}'],
               s=10, alpha=0.5, label=gene)
ax.axhline(0, color='#ccc', lw=0.8)
ax.axvline(0, color='#ccc', lw=0.8)
ax.set_xlabel(f'observed log2FC ({SPEC_X.name})')
ax.set_ylabel(f'observed log2FC ({SPEC_Y.name})')
ax.legend(frameon=False, fontsize=8, title='cis gene')
ax.set_title(f'All cis genes: {SPEC_X.name} vs {SPEC_Y.name}')
fig.savefig(os.path.join(PLOT_DIR, f'ALL_{SPEC_X.name}_vs_{SPEC_Y.name}_obs_log2fc.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## Grid of pairwise comparisons (all three datasets)

Adds Domingo into the mix: for a cis gene all three have a completed fit for (GFI1B, NFE2), this produces one grid per parameter -- 3 subplots (Domingo-Morris, Domingo-Replogle, Morris-Replogle), each colored/sorted by `observed_log2fc` as before.

**Comparisons against Domingo will have far fewer points than Morris-vs-Replogle** -- Domingo's own trans panel is only ~91 genes (vs. Morris/Replogle's transcriptome-wide ~20k), so this is expected, not a bug.

Each pair's join key is chosen independently: Morris-vs-Replogle merges on Ensembl `gene_id` (unambiguous); any pair involving Domingo falls back to `gene_symbol` (Domingo carries no Ensembl mapping) -- see `trans_param_compare.py`'s `merge_pair()` docstring.

In [ ]:
GRID_CIS_GENE = 'GFI1B'
GRID_SPECS = [DOMINGO, MORRIS, REPLOGLE]

dfs, figs = compare_cis_gene_grid(
    GRID_SPECS, GRID_CIS_GENE,
    out_dir=os.path.join(PLOT_DIR, 'pairwise_grids'),
)
print(f'Produced grids for: {list(figs)}')

In [ ]:
# Inline preview of one parameter's grid
figs['observed_log2fc']

### All cis genes at once

Automates the above across every cis gene in `DOMINGO.cis_genes` (GFI1B, NFE2, MYB, TET2), using whichever of {Domingo, Morris, Replogle} actually lists that gene as fit (Morris drops out for MYB/TET2 automatically -- structural, not an error). Raises immediately if a dataset that DOES claim a gene turns out to be missing its summary CSV -- run the Backfill section above first if that happens.

In [ ]:
all_results = compare_all_cis_genes_grid(
    [DOMINGO, MORRIS, REPLOGLE],
    out_dir=os.path.join(PLOT_DIR, 'pairwise_grids'),
)
print(f'Produced grids for cis genes: {list(all_results)}')

## Customising further

- Swap `SPEC_X`/`SPEC_Y` for any pair in `comparative.datasets.ALL_DATASETS` (or add a new `DatasetSpec` there for a new run).
- `observed_log2fc`/`full_log2fc` are signed (not the raw unsigned `log2(y_max/y_min)` magnitude bayesDREAM's own `save_trans_summary()` writes to the CSV) -- `load_trans_summary()` multiplies both by the net direction of the fitted curve over the observed x range. The CSV on disk is untouched; this is an in-memory transformation, every time a summary is loaded.
- Default coloring (`plot_param_grid`/`plot_pairwise_grid`/`compare_cis_gene*`) is `dependency_category` -- categorical, one color per {dependent in A only, dependent in B only, dependent in both} with a proper legend, and genes dependent in *neither* dropped from the plot (and its correlation stats) entirely via `exclude_categories=('neither',)`. Pass `exclude_categories=None` to keep them (shown in a 4th color), or a numeric `PARAM_ALIASES` name (e.g. `color_by_param='min_abs_observed_log2fc'`, or `color_by_param='observed_log2fc', color_dataset=SPEC_Y.name` for the old single-dataset behavior) for continuous coloring instead -- `exclude_categories` is then a no-op.
- `plot_param_grid(..., color_by_param=None)` turns off coloring (plain gray points, all genes shown).
- `scatter_param`'s correlation annotation now includes each of Pearson r / Spearman ρ's own p-value alongside the coefficient.
- `DEFAULT_GRID_PARAMS` correlates `log2_y_ntc` (NTC baseline expression) instead of `Vmax_a` (each model's own fitted Hill amplitude, not directly comparable across independently-fit models).
- `plot_param_grid_by_dependency(merged, SPEC_X.name, SPEC_Y.name, CIS_GENE, params=GRID_PARAMS)` -- the 4-row (dependent in X only / Y only / both / neither) version demoed above, keeping all 4 groups instead of dropping "neither" -- also works as a drop-in wherever `plot_param_grid` does. Colors by `min_abs_observed_log2fc` by default (not `dependency_category` -- redundant with the row split itself).
- `plot_pairwise_grid`/`compare_cis_gene_grid`/`compare_all_cis_genes_grid`/`plot_param_grid_by_dependency` all default `shared_lims=True`: every panel a given dataset appears in shares the same axis range (computed from that dataset's FULL values, not just whatever subset of genes it happens to share with one particular partner/category). Without this, e.g. Domingo's axis in a Domingo-vs-Morris panel and Domingo-vs-Replogle panel would differ, since each pair only shares a different ~70-90 gene subset of Domingo's ~91-gene panel -- pass `shared_lims=False` for the old independently-scaled-per-panel behavior.
- Pass `highlight_col=None` to any of the plotting functions to drop the `is_dependent` outline markers (only relevant with non-categorical coloring -- categorical dependency coloring already encodes this and skips the redundant outline automatically).
- `comparative/trans_param_compare.py`'s `PARAM_ALIASES` dict lists every parameter name recognised by `scatter_param`/`plot_param_grid` -- add an entry there for any column you want to compare that isn't already covered.
- `plot_pairwise_grid` picks `color_dataset` as whichever dataset comes first alphabetically in each pair by default (only relevant when `color_by_param` is a per-dataset logical name, not the cross-dataset `dependency_category`/`min_abs_observed_log2fc` defaults) -- pass `color_dataset=` explicitly to `compare_cis_gene_grid`/`plot_pairwise_grid` to fix it to one dataset across all panels.
- `y_log2fc_at_xm1` (log2FC in y at 50% cis-gene knockdown, matching `examples/vignette_trans_fit_crispri.py`'s `HILL_LOG2FC_TARGETS`) only shows up once a dataset's trans_feature_summary CSV has been backfilled with it -- see `comparative/reconstruct_export.py` (Domingo/Morris) or the Replogle notebook snippet.